# Audio Preprocessing Inspection Notebook

This notebook loads one audio file, computes a log-band feature representation, extracts sparse salient events, and visualizes the results with Plotly. It is designed for interactive inspection of the preprocessing pipeline rather than model training.

The final 3D plot is a sparse event cloud where each retained point represents one salient feature cell, with `x = frequency band`, `y = feature channel`, and `z = frame within chunk`.


In [ ]:
# Cell 2 - Imports

from pathlib import Path

import librosa
import librosa.display
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import scipy
import ipywidgets as widgets

from IPython.display import HTML, Markdown, display
from ipyfilechooser import FileChooser


In [ ]:
# Cell 3 - Configuration

# Audio loading
TARGET_SR = 16000
MAX_AUDIO_SECONDS = None
AUDIO_OFFSET_SECONDS = 0.0
MONO = True
AUDIO_FILE_PATH = None  # Optional fallback path if the file chooser does not render in your frontend.
FILE_CHOOSER_START_DIR = "datasets"

# Spectral frontend
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
WINDOW = "hann"

# Log-band projection
NUM_BANDS = 96
FMIN = 50.0
FMAX = 8000.0

# Handcrafted feature channels
FEATURES_ENABLED = ["log_energy", "slope", "curvature", "peakness"]
FEATURE_NAME_ORDER = FEATURES_ENABLED.copy()

PEAKNESS_NEIGHBOURHOOD = 1
EPSILON = 1e-8

# Feature normalization and salience
NORMALIZE_FEATURES = True
NORMALIZATION_MODE = "robust_per_feature"

# Chunk selection for sparse visualization
FRAMES_PER_CHUNK = 32
CHUNK_START_FRAME = 0
SELECTED_FRAME_IN_CHUNK = 0

# Sparsification controls
TOP_K_PER_FRAME = 24
SALIENCE_MODE = "abs_robust_zscore"
SALIENCE_THRESHOLD = None

# Plot controls
PLOT_MARKER_SIZE = 5
PLOT_MARKER_OPACITY = 0.8
PLOT_COLOR_MODE = "salience"   # or "value"
PLOT_TEMPLATE = "plotly_white"

# File selection
SUPPORTED_EXTENSIONS = [".wav", ".mp3", ".flac", ".ogg", ".m4a"]

# Derived values
NUM_FEATURES = len(FEATURE_NAME_ORDER)
TOTAL_CELLS_PER_FRAME = NUM_BANDS * NUM_FEATURES


`HOP_LENGTH` controls the time resolution of the analysis: smaller hops give finer temporal detail, while larger hops smooth things over. `FRAMES_PER_CHUNK` controls how long the 3D event cloud is along the time axis.

`NUM_BANDS` sets the vertical frequency resolution after projection into log-spaced bands. `TOP_K_PER_FRAME` sets how sparse each frame becomes by keeping only the most salient cells.

`FEATURES_ENABLED` defines which handcrafted feature channels are computed, and those channels become the y-axis categories in the final 3D sparse event cloud.


In [ ]:
# Cell 5 - File Chooser

display(Markdown("### Select one audio file to inspect"))
display(Markdown(
    "Pick a file with the chooser below. If `ipyfilechooser` does not render in your frontend, set `AUDIO_FILE_PATH` in Cell 3 and rerun from Cell 7."
))

chooser_start_dir = Path(FILE_CHOOSER_START_DIR).expanduser()
if not chooser_start_dir.is_absolute():
    chooser_start_dir = (Path.cwd() / chooser_start_dir).resolve()
if not chooser_start_dir.exists():
    chooser_start_dir = Path.cwd()

file_chooser = FileChooser(path=str(chooser_start_dir))
file_chooser.title = "Choose one audio file"
file_chooser.show_hidden = False
file_chooser.use_dir_icons = True
file_chooser.filter_pattern = [f"*{ext}" for ext in SUPPORTED_EXTENSIONS]

if AUDIO_FILE_PATH:
    preset_path = Path(AUDIO_FILE_PATH).expanduser()
    if not preset_path.is_absolute():
        preset_path = (Path.cwd() / preset_path).resolve()
    if preset_path.exists():
        file_chooser.reset(path=str(preset_path.parent), filename=preset_path.name)

selected_file_preview = widgets.HTML(value="<i>No file selected yet.</i>")


def _refresh_selected_file_preview(chooser):
    selected = getattr(chooser, "selected", None)
    if selected:
        selected_file_preview.value = f"<b>Selected file:</b> {selected}"
    elif AUDIO_FILE_PATH:
        selected_file_preview.value = f"<b>Configured AUDIO_FILE_PATH:</b> {AUDIO_FILE_PATH}"
    else:
        selected_file_preview.value = "<i>No file selected yet.</i>"


file_chooser.register_callback(_refresh_selected_file_preview)
_refresh_selected_file_preview(file_chooser)

display(Markdown(f"Chooser start directory: `{chooser_start_dir}`"))
display(file_chooser)
display(selected_file_preview)


The next cell loads the currently selected file and prints summary information about it. This notebook processes exactly one chosen file at a time.


In [ ]:
# Cell 7 - Load Audio

def _resolve_audio_path(chooser, configured_path=None):
    """Resolve the audio path from the chooser selection or a config fallback."""
    if configured_path is not None and str(configured_path).strip():
        configured = Path(str(configured_path)).expanduser()
        if not configured.is_absolute():
            configured = (Path.cwd() / configured).resolve()
        return str(configured)

    selected = getattr(chooser, "selected", None)
    if selected:
        return str(Path(selected).expanduser())

    selected_path = getattr(chooser, "selected_path", None)
    selected_filename = getattr(chooser, "selected_filename", None)
    if selected_path and selected_filename:
        return str(Path(selected_path) / selected_filename)

    raise ValueError(
        "No file selected. Use Cell 5 to pick one audio file, or set AUDIO_FILE_PATH in Cell 3 if the chooser does not render in your frontend."
    )



def load_audio(file_path, target_sr, mono=True, offset_seconds=0.0, max_seconds=None):
    """Load one audio file with librosa and return waveform, sample rate, and metadata."""
    path = Path(file_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Selected file does not exist: {path}")

    if path.suffix.lower() not in {ext.lower() for ext in SUPPORTED_EXTENSIONS}:
        supported = ", ".join(SUPPORTED_EXTENSIONS)
        raise ValueError(f"Unsupported file extension '{path.suffix}'. Supported extensions: {supported}")

    y, sr = librosa.load(
        path,
        sr=target_sr,
        mono=mono,
        offset=offset_seconds,
        duration=max_seconds,
    )

    num_samples = y.shape[-1] if y.ndim > 1 else len(y)
    duration_seconds = num_samples / float(sr)
    channel_description = "mono" if y.ndim == 1 else f"{y.shape[0]} channels"

    metadata = {
        "file_path": str(path),
        "sample_rate": sr,
        "duration_seconds": duration_seconds,
        "num_samples": num_samples,
        "channel_description": channel_description,
        "mono_requested": mono,
    }

    print(f"File path: {metadata['file_path']}")
    print(f"Sample rate: {metadata['sample_rate']} Hz")
    print(f"Duration: {metadata['duration_seconds']:.3f} s")
    print(f"Number of samples: {metadata['num_samples']}")
    print(f"Mono/stereo result: {metadata['channel_description']}")

    return y, sr, metadata



audio_path = _resolve_audio_path(file_chooser, AUDIO_FILE_PATH)
y, sr, audio_metadata = load_audio(
    audio_path,
    target_sr=TARGET_SR,
    mono=MONO,
    offset_seconds=AUDIO_OFFSET_SECONDS,
    max_seconds=MAX_AUDIO_SECONDS,
)
audio_duration_seconds = audio_metadata["duration_seconds"]


Next, the notebook computes an STFT magnitude representation and projects it into log-frequency bands. This is usually a more stable and interpretable intermediate for the intended sparse spatiotemporal representation than the raw waveform.


In [ ]:
# Cell 9 - Spectral Frontend

def compute_stft_magnitude(y, n_fft, hop_length, win_length, window):
    """Compute an STFT magnitude spectrogram with shape (freq_bins, num_frames)."""
    y_mono = librosa.to_mono(y) if getattr(y, "ndim", 1) > 1 else y
    stft_complex = librosa.stft(
        y_mono,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
    )
    return np.abs(stft_complex)


def build_log_band_filters(sr, n_fft, num_bands, fmin, fmax):
    """Build a simple triangular filterbank with log-spaced center frequencies."""
    fft_freqs = np.fft.rfftfreq(n_fft, d=1.0 / sr)
    valid_fmax = min(float(fmax), float(sr) / 2.0)
    valid_fmin = max(float(fmin), fft_freqs[1] if len(fft_freqs) > 1 else float(fmin))
    if valid_fmin >= valid_fmax:
        raise ValueError("fmin must be smaller than fmax after accounting for the sample rate.")

    centers = np.geomspace(valid_fmin, valid_fmax, num=num_bands)
    boundaries = np.empty(num_bands + 1, dtype=float)
    boundaries[0] = valid_fmin
    boundaries[-1] = valid_fmax
    if num_bands > 1:
        boundaries[1:-1] = np.sqrt(centers[:-1] * centers[1:])

    filters = np.zeros((num_bands, len(fft_freqs)), dtype=float)
    for band_idx in range(num_bands):
        left = boundaries[band_idx]
        center = centers[band_idx]
        right = boundaries[band_idx + 1]

        left_mask = (fft_freqs >= left) & (fft_freqs <= center)
        right_mask = (fft_freqs >= center) & (fft_freqs <= right)

        if center > left:
            filters[band_idx, left_mask] = (fft_freqs[left_mask] - left) / (center - left)
        if right > center:
            filters[band_idx, right_mask] = (right - fft_freqs[right_mask]) / (right - center)

        band_sum = filters[band_idx].sum()
        if band_sum > 0:
            filters[band_idx] /= band_sum

    return filters


def project_to_log_bands(stft_mag, band_filters, eps=1e-8):
    """Project STFT magnitude into log-spaced band energies with shape (num_bands, num_frames)."""
    power_spectrogram = stft_mag ** 2
    band_energy = band_filters @ power_spectrogram
    return np.maximum(band_energy, eps)


stft_mag = compute_stft_magnitude(y, N_FFT, HOP_LENGTH, WIN_LENGTH, WINDOW)
band_filters = build_log_band_filters(sr, N_FFT, NUM_BANDS, FMIN, FMAX)
band_energy = project_to_log_bands(stft_mag, band_filters, eps=EPSILON)

print(f"STFT magnitude shape: {stft_mag.shape} (freq_bins, num_frames)")
print(f"Band filter shape: {band_filters.shape} (num_bands, freq_bins)")
print(f"Band energy shape: {band_energy.shape} (num_bands, num_frames)")


In [ ]:
# Cell 10 - Plot Waveform

def display_plotly_figure(fig):
    """Display a Plotly figure as inline HTML without relying on prior import state."""
    from IPython.display import HTML, display

    html = fig.to_html(full_html=False, include_plotlyjs=True, config={"responsive": True})
    display(HTML(html))



def plot_waveform(y, sr, template="plotly_white"):
    """Plot one waveform with Plotly using time in seconds on the x-axis."""
    y_plot = librosa.to_mono(y) if getattr(y, "ndim", 1) > 1 else y
    time_axis = np.arange(len(y_plot)) / float(sr)
    title_name = Path(audio_path).name if "audio_path" in globals() else "Selected audio"

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=time_axis,
            y=y_plot,
            mode="lines",
            name="waveform",
            line=dict(width=1.0),
        )
    )
    fig.update_layout(
        template=template,
        title=f"Waveform: {title_name}",
        xaxis_title="Time (seconds)",
        yaxis_title="Amplitude",
        height=350,
    )
    display_plotly_figure(fig)
    return fig


_waveform_fig = plot_waveform(y, sr, template=PLOT_TEMPLATE)


In [ ]:
# Cell 11 - Plot Band Heatmap

def plot_band_heatmap(band_energy, sr, hop_length, template="plotly_white"):
    """Plot projected band energy as a heatmap with time on x and band index on y."""
    num_frames = band_energy.shape[1]
    time_axis = np.arange(num_frames) * hop_length / float(sr)
    energy_for_plot = np.log10(band_energy + EPSILON)

    fig = go.Figure(
        data=go.Heatmap(
            z=energy_for_plot,
            x=time_axis,
            y=np.arange(band_energy.shape[0]),
            colorscale="Viridis",
            colorbar=dict(title="log10 energy"),
        )
    )
    fig.update_layout(
        template=template,
        title="Log-band energy heatmap",
        xaxis_title="Time (seconds)",
        yaxis_title="Band index",
        height=450,
    )
    display_plotly_figure(fig)
    return fig


_band_heatmap_fig = plot_band_heatmap(band_energy, sr, HOP_LENGTH, template=PLOT_TEMPLATE)


For each frame, the notebook builds a dense `(num_bands, num_features)` feature grid using four handcrafted channels. `log energy` is the bandwise log magnitude, `slope` is the first difference across bands, `curvature` is the second difference across bands, and `peakness` is a simple local prominence against neighboring bands.


In [ ]:
# Cell 13 - Feature Computation

def compute_log_energy_feature(band_energy, eps=1e-8):
    """Compute log energy for each band and frame."""
    return np.log(band_energy + eps)


def compute_slope_feature(log_energy):
    """Compute a first finite difference across the band axis."""
    slope = np.zeros_like(log_energy)
    if log_energy.shape[0] == 1:
        return slope

    slope[0] = log_energy[1] - log_energy[0]
    slope[-1] = log_energy[-1] - log_energy[-2]
    if log_energy.shape[0] > 2:
        slope[1:-1] = 0.5 * (log_energy[2:] - log_energy[:-2])
    return slope


def compute_curvature_feature(log_energy):
    """Compute a second finite difference across the band axis."""
    curvature = np.zeros_like(log_energy)
    if log_energy.shape[0] > 2:
        curvature[1:-1] = log_energy[2:] - (2.0 * log_energy[1:-1]) + log_energy[:-2]
        curvature[0] = curvature[1]
        curvature[-1] = curvature[-2]
    return curvature


def compute_peakness_feature(log_energy, neighbourhood=1):
    """Compute local prominence relative to neighboring bands."""
    if neighbourhood < 1:
        raise ValueError("peakness neighbourhood must be at least 1")

    peakness = np.zeros_like(log_energy)
    num_bands = log_energy.shape[0]
    for band_idx in range(num_bands):
        left = max(0, band_idx - neighbourhood)
        right = min(num_bands, band_idx + neighbourhood + 1)
        neighborhood_values = log_energy[left:right]
        center_position = band_idx - left
        neighbors = np.delete(neighborhood_values, center_position, axis=0)
        if neighbors.size == 0:
            continue
        peakness[band_idx] = log_energy[band_idx] - neighbors.mean(axis=0)
    return peakness


def compute_feature_grid(band_energy, features_enabled, peakness_neighbourhood=1, eps=1e-8):
    """Build a dense feature tensor with shape (num_frames, num_bands, num_features)."""
    log_energy = compute_log_energy_feature(band_energy, eps=eps)
    feature_map = {
        "log_energy": log_energy,
        "slope": compute_slope_feature(log_energy),
        "curvature": compute_curvature_feature(log_energy),
        "peakness": compute_peakness_feature(log_energy, neighbourhood=peakness_neighbourhood),
    }

    feature_names = []
    feature_planes = []
    for feature_name in features_enabled:
        if feature_name not in feature_map:
            available = ", ".join(sorted(feature_map))
            raise ValueError(f"Unknown feature '{feature_name}'. Available features: {available}")
        feature_names.append(feature_name)
        feature_planes.append(feature_map[feature_name].T)

    feature_grid = np.stack(feature_planes, axis=-1)
    return feature_grid, feature_names


feature_grid, feature_names = compute_feature_grid(
    band_energy,
    features_enabled=FEATURE_NAME_ORDER,
    peakness_neighbourhood=PEAKNESS_NEIGHBOURHOOD,
    eps=EPSILON,
)

print(f"Feature grid shape: {feature_grid.shape} (num_frames, num_bands, num_features)")


Normalization makes different feature channels more comparable so one channel does not dominate just because of scale. Salience is a first-pass score for how interesting each cell is before sparsification, so the later top-k selection can focus on the strongest local structure.


In [ ]:
# Cell 15 - Normalization And Salience

def robust_zscore(x, axis=None, eps=1e-8):
    """Compute a robust z-score using the median and MAD."""
    median = np.median(x, axis=axis, keepdims=True)
    mad = np.median(np.abs(x - median), axis=axis, keepdims=True)
    scale = 1.4826 * mad
    return (x - median) / (scale + eps)


def normalize_feature_grid(feature_grid, mode="robust_per_feature", eps=1e-8):
    """Normalize features across frames and bands, independently per feature channel."""
    if mode in (None, "none"):
        return feature_grid.copy()
    if mode == "robust_per_feature":
        return robust_zscore(feature_grid, axis=(0, 1), eps=eps)
    raise ValueError(f"Unsupported normalization mode: {mode}")


def compute_salience(feature_grid_normalized, mode="abs_robust_zscore"):
    """Compute one salience score per cell with shape (num_frames, num_bands, num_features)."""
    if mode == "abs_robust_zscore":
        return np.abs(feature_grid_normalized)
    if mode == "absolute_value":
        return np.abs(feature_grid_normalized)
    raise ValueError(f"Unsupported salience mode: {mode}")


feature_grid_normalized = (
    normalize_feature_grid(feature_grid, mode=NORMALIZATION_MODE, eps=EPSILON)
    if NORMALIZE_FEATURES
    else feature_grid.copy()
)
salience_grid = compute_salience(feature_grid_normalized, mode=SALIENCE_MODE)

print(f"Normalized feature grid shape: {feature_grid_normalized.shape}")
print(f"Salience grid shape: {salience_grid.shape}")


In [ ]:
# Cell 16 - Plot Feature Slice

def plot_feature_slice(feature_grid_or_normalized, feature_names, frame_idx, title_prefix="", template="plotly_white"):
    """Plot one frame slice as a dense (bands x features) heatmap."""
    num_frames = feature_grid_or_normalized.shape[0]
    if frame_idx < 0 or frame_idx >= num_frames:
        raise IndexError(f"Frame index {frame_idx} is out of range for {num_frames} frames.")

    frame_slice = feature_grid_or_normalized[frame_idx]
    title = f"{title_prefix} frame {frame_idx}".strip()

    fig = go.Figure(
        data=go.Heatmap(
            z=frame_slice,
            x=feature_names,
            y=np.arange(frame_slice.shape[0]),
            colorscale="RdBu",
            colorbar=dict(title="value"),
        )
    )
    fig.update_layout(
        template=template,
        title=title,
        xaxis_title="Feature channel",
        yaxis_title="Band index",
        height=450,
    )
    display_plotly_figure(fig)
    return fig


absolute_frame = CHUNK_START_FRAME + SELECTED_FRAME_IN_CHUNK
_feature_slice_fig = plot_feature_slice(
    feature_grid_normalized,
    feature_names,
    frame_idx=absolute_frame,
    title_prefix="Normalized feature slice,",
    template=PLOT_TEMPLATE,
)


Each frame contains a dense `(bands x features)` grid. Sparsification keeps only the top-k most salient cells per frame, turning that dense representation into a compact sparse event set that is easier to inspect over time.


In [ ]:
# Cell 18 - Sparse Event Selection

def select_top_k_events_for_frame(values_2d, salience_2d, feature_names, absolute_frame, chunk_frame, top_k, threshold=None):
    """Select the top-k salient cells from one (bands x features) frame."""
    if values_2d.shape != salience_2d.shape:
        raise ValueError("values_2d and salience_2d must have the same shape.")

    num_bands, num_features = values_2d.shape
    total_cells = num_bands * num_features
    if len(feature_names) != num_features:
        raise ValueError("feature_names length must match the feature dimension.")

    if top_k < 0:
        raise ValueError("top_k must be non-negative.")
    if total_cells == 0 or top_k == 0:
        return []

    salience_flat = salience_2d.reshape(-1)
    value_flat = values_2d.reshape(-1)
    if threshold is not None:
        candidate_indices = np.flatnonzero(salience_flat >= threshold)
    else:
        candidate_indices = np.arange(total_cells)

    if candidate_indices.size == 0:
        return []

    k = min(int(top_k), int(candidate_indices.size), int(total_cells))
    candidate_salience = salience_flat[candidate_indices]
    if k == candidate_indices.size:
        top_indices = candidate_indices[np.argsort(candidate_salience)[::-1]]
    else:
        partial = np.argpartition(candidate_salience, -k)[-k:]
        top_indices = candidate_indices[partial]
        top_indices = top_indices[np.argsort(salience_flat[top_indices])[::-1]]

    events = []
    for flat_idx in top_indices:
        band_idx, feature_idx = np.unravel_index(flat_idx, values_2d.shape)
        events.append(
            {
                "absolute_frame": int(absolute_frame),
                "chunk_frame": int(chunk_frame),
                "band_idx": int(band_idx),
                "feature_idx": int(feature_idx),
                "feature_name": feature_names[feature_idx],
                "value": float(value_flat[flat_idx]),
                "salience": float(salience_flat[flat_idx]),
            }
        )
    return events


def build_sparse_events(feature_grid, salience_grid, feature_names, chunk_start_frame, frames_per_chunk, top_k, threshold=None):
    """Build a sparse event table for one contiguous chunk of frames."""
    if feature_grid.shape != salience_grid.shape:
        raise ValueError("feature_grid and salience_grid must have matching shapes.")

    total_frames = feature_grid.shape[0]
    if chunk_start_frame < 0 or chunk_start_frame >= total_frames:
        raise IndexError(
            f"chunk_start_frame {chunk_start_frame} is out of range for {total_frames} frames."
        )
    if frames_per_chunk <= 0:
        raise ValueError("frames_per_chunk must be positive.")

    chunk_end_frame = min(chunk_start_frame + frames_per_chunk, total_frames)
    rows = []
    for absolute_frame in range(chunk_start_frame, chunk_end_frame):
        chunk_frame = absolute_frame - chunk_start_frame
        rows.extend(
            select_top_k_events_for_frame(
                feature_grid[absolute_frame],
                salience_grid[absolute_frame],
                feature_names,
                absolute_frame=absolute_frame,
                chunk_frame=chunk_frame,
                top_k=top_k,
                threshold=threshold,
            )
        )

    columns = [
        "absolute_frame",
        "chunk_frame",
        "band_idx",
        "feature_idx",
        "feature_name",
        "value",
        "salience",
    ]
    return pd.DataFrame(rows, columns=columns)


sparse_events_df = build_sparse_events(
    feature_grid=feature_grid_normalized,
    salience_grid=salience_grid,
    feature_names=feature_names,
    chunk_start_frame=CHUNK_START_FRAME,
    frames_per_chunk=FRAMES_PER_CHUNK,
    top_k=TOP_K_PER_FRAME,
    threshold=SALIENCE_THRESHOLD,
)

frames_used = min(FRAMES_PER_CHUNK, feature_grid.shape[0] - CHUNK_START_FRAME)
print(f"Frames used: {frames_used}")
print(f"Total sparse events: {len(sparse_events_df)}")
if sparse_events_df.empty:
    print("Sparse event table is empty. Try lowering the threshold or increasing top-k.")
else:
    display(sparse_events_df.head())


In [ ]:
# Cell 19 - Plot Sparse Frame Events

def plot_sparse_frame_events(sparse_events_df, frame_in_chunk, color_mode="salience", template="plotly_white"):
    """Plot retained sparse events for one frame as a 2D scatter plot."""
    if sparse_events_df.empty:
        raise ValueError("Sparse event dataframe is empty. There is nothing to plot.")

    available_frames = sorted(sparse_events_df["chunk_frame"].unique().tolist())
    if frame_in_chunk not in available_frames:
        raise IndexError(
            f"frame_in_chunk {frame_in_chunk} is not available in the sparse chunk. "
            f"Available chunk frames: {available_frames}"
        )

    frame_df = sparse_events_df[sparse_events_df["chunk_frame"] == frame_in_chunk].copy()
    color_column = "salience" if color_mode == "salience" else "value"

    customdata = np.stack(
        [
            frame_df["feature_name"],
            frame_df["absolute_frame"],
            frame_df["value"],
            frame_df["salience"],
        ],
        axis=-1,
    )

    fig = go.Figure(
        data=go.Scatter(
            x=frame_df["feature_idx"],
            y=frame_df["band_idx"],
            mode="markers",
            marker=dict(
                size=PLOT_MARKER_SIZE + 2,
                opacity=PLOT_MARKER_OPACITY,
                color=frame_df[color_column],
                colorscale="Turbo",
                colorbar=dict(title=color_column),
            ),
            customdata=customdata,
            hovertemplate=(
                "Chunk frame: %{text}<br>"
                "Absolute frame: %{customdata[1]}<br>"
                "Band index: %{y}<br>"
                "Feature index: %{x}<br>"
                "Feature name: %{customdata[0]}<br>"
                "Value: %{customdata[2]:.4f}<br>"
                "Salience: %{customdata[3]:.4f}<extra></extra>"
            ),
            text=frame_df["chunk_frame"],
        )
    )
    fig.update_layout(
        template=template,
        title=f"Sparse events for chunk frame {frame_in_chunk}",
        xaxis_title="Feature channel",
        yaxis_title="Band index",
        height=450,
    )
    fig.update_xaxes(tickmode="array", tickvals=list(range(len(feature_names))), ticktext=feature_names)
    display_plotly_figure(fig)
    return fig


_sparse_frame_fig = plot_sparse_frame_events(
    sparse_events_df,
    frame_in_chunk=SELECTED_FRAME_IN_CHUNK,
    color_mode=PLOT_COLOR_MODE,
    template=PLOT_TEMPLATE,
)


Each point in the 3D plot is one retained salient feature cell. The axes are `x = frequency band`, `y = feature channel`, and `z = frame index within chunk`.

Visually, look for stable rails or ridges, short bursts, clusters, sloped traces, or diffuse noise-like scatter. Those patterns help show whether the sparse representation is preserving structure or just producing random glitter.


In [ ]:
# Cell 21 - Plot Sparse Chunk 3D

def plot_sparse_chunk_3d(sparse_events_df, color_mode="salience", marker_size=5, marker_opacity=0.8, template="plotly_white"):
    """Plot the sparse event cloud for one chunk as a 3D scatter plot."""
    if sparse_events_df.empty:
        raise ValueError("Sparse event dataframe is empty. There is nothing to plot.")

    color_column = "salience" if color_mode == "salience" else "value"
    customdata = np.stack(
        [
            sparse_events_df["chunk_frame"],
            sparse_events_df["absolute_frame"],
            sparse_events_df["feature_name"],
            sparse_events_df["value"],
            sparse_events_df["salience"],
        ],
        axis=-1,
    )

    fig = go.Figure(
        data=go.Scatter3d(
            x=sparse_events_df["band_idx"],
            y=sparse_events_df["feature_idx"],
            z=sparse_events_df["chunk_frame"],
            mode="markers",
            marker=dict(
                size=marker_size,
                opacity=marker_opacity,
                color=sparse_events_df[color_column],
                colorscale="Turbo",
                colorbar=dict(title=color_column),
            ),
            customdata=customdata,
            hovertemplate=(
                "Chunk frame: %{customdata[0]}<br>"
                "Absolute frame: %{customdata[1]}<br>"
                "Band index: %{x}<br>"
                "Feature index: %{y}<br>"
                "Feature name: %{customdata[2]}<br>"
                "Value: %{customdata[3]:.4f}<br>"
                "Salience: %{customdata[4]:.4f}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        template=template,
        title="3D sparse event cloud for selected chunk",
        scene=dict(
            xaxis_title="Frequency band index",
            yaxis_title="Feature channel index",
            zaxis_title="Frame index within chunk",
            yaxis=dict(
                tickmode="array",
                tickvals=list(range(len(feature_names))),
                ticktext=feature_names,
            ),
        ),
        height=700,
    )
    display_plotly_figure(fig)
    return fig


_sparse_chunk_fig = plot_sparse_chunk_3d(
    sparse_events_df,
    color_mode=PLOT_COLOR_MODE,
    marker_size=PLOT_MARKER_SIZE,
    marker_opacity=PLOT_MARKER_OPACITY,
    template=PLOT_TEMPLATE,
)


In [ ]:
# Cell 22 - Summary Footer

selected_chunk_end = min(CHUNK_START_FRAME + FRAMES_PER_CHUNK, feature_grid.shape[0])
frames_in_chunk = max(0, selected_chunk_end - CHUNK_START_FRAME)
average_events_per_frame = len(sparse_events_df) / frames_in_chunk if frames_in_chunk else 0.0

print("Quick summary")
print(f"File name: {Path(audio_path).name}")
print(f"Sample rate: {sr} Hz")
print(f"Band-energy shape: {band_energy.shape}")
print(f"Feature-grid shape: {feature_grid.shape}")
print(f"Selected chunk frame range: [{CHUNK_START_FRAME}, {selected_chunk_end - 1}]")
print(f"Sparse event count: {len(sparse_events_df)}")
print(f"Average events per frame: {average_events_per_frame:.2f}")
print(f"Selected feature names: {feature_names}")
